# vLLM 실제로 돌려보기 (Colab)

지금까지는 커널 하나만 따로 떼어서 봤다면, 이번에는 vLLM을 통째로 설치하고 작은 모델을 직접 돌려서, "요청 → 스케줄러 → attention 커널"이라는 전체 파이프라인이 로그로 어떻게 찍히는지 직접 확인해봅니다.

**런타임을 GPU로 설정하세요** (메뉴 → 런타임 → 런타임 유형 변경 → GPU). 무료 T4는 VRAM 16GB라 작은 모델(1~2B급)만 돌릴 수 있습니다.

**설치가 약간 걸립니다** (보통 3~6분) — 의존성이 많아서 그렇습니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q vllm
# vllm이 새 PyTorch(CUDA 버전 다름)를 깔면서, Colab에 미리 깔려있던
# torchaudio/torchvision과 CUDA 버전이 어긋나 import 시점에 RuntimeError가
# 날 수 있습니다. 이 노트북은 텍스트 전용이라 둘 다 필요 없으니 지웁니다.
!pip uninstall -y -q torchaudio torchvision

## 1. 작은 모델 로드

`TinyLlama-1.1B-Chat`을 씁니다 — 우리가 소스코드로 추적했던 `LlamaAttention` 경로를 그대로 타는 Llama 아키텍처 모델이고, 게이트 없는 라이선스라 바로 다운로드됩니다.

vLLM이 모델을 초기화하는 동안 나오는 로그가 많은데, 그 안에 우리가 지난번에 코드로 봤던 "어떤 attention 백엔드를 고르는지", "KV 캐시를 어떻게 잡는지" 같은 줄들이 있습니다. `profiler_config`로 나중에(4번) 쓸 프로파일러 출력 경로도 미리 지정해둡니다.

In [ ]:
import os

# 엔진 코어를 별도 서브프로세스로 안 띄우고 이 노트북 프로세스 안에서 그대로
# 돌립니다. Jupyter/Colab처럼 이미 CUDA가 초기화된 프로세스에서 fork로 서브
# 프로세스를 새로 띄우면 깨지는 경우가 많아서, 이 값을 꺼서 우리가 지난번에
# 코드로 봤던 InprocClient 경로를 강제로 타게 합니다.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# vLLM은 GPU 워커 초기화(NCCL 프로세스 그룹 생성) 때 stdout을 잠깐 숨기려고
# sys.stdout.fileno()를 호출하는데, Jupyter/Colab의 stdout은 ipykernel이 만든
# 가짜 스트림이라 fileno()가 없어서 `io.UnsupportedOperation: fileno`로 죽습니다.
# vllm/utils/system_utils.py의 suppress_stdout()이 VLLM_LOGGING_LEVEL=DEBUG일
# 때는 이 fileno() 호출 자체를 건너뛰도록 이미 만들어져 있어서, 이걸로 우회합니다.
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

from vllm import LLM, SamplingParams

os.makedirs("/content/vllm_trace", exist_ok=True)

llm = LLM(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    # 4번 스텝의 프로파일러가 여기서 만든 트레이스 폴더를 씁니다.
    profiler_config={"profiler": "torch", "torch_profiler_dir": "/content/vllm_trace"},
)

## 2. 위 출력에서 attention 백엔드 찾기

바로 위 셀을 실행한 로그 전체를 위로 스크롤해서 `backend`나 `Attn`이 들어간 줄을 찾아보세요. 우리가 `vllm/v1/attention/backends/registry.py`에서 봤던 `TRITON_ATTN`, `FLASH_ATTN` 같은 이름이 그대로 보일 겁니다. (Colab 무료 T4는 구세대 아키텍처라 FlashAttention이 안 되는 경우가 많고, 그럴 때 vLLM이 자동으로 Triton 백엔드로 폴백합니다 — 우리가 지난번에 직접 뜯어본 바로 그 커널입니다.)

In [ ]:
# 엔진이 실제 사용 중인 attention backend를 코드로도 확인
attn_layer = None
for name, module in llm.llm_engine.model_executor.driver_worker.model_runner.model.named_modules():
    cls_name = type(module).__name__
    if cls_name == "Attention":
        attn_layer = module
        break

if attn_layer is not None:
    print("attn backend:", attn_layer.attn_backend.get_name())
    print("impl class  :", type(attn_layer.impl).__name__)
else:
    print("Attention 레이어를 모델 모듈 트리에서 못 찾았습니다 (vLLM 버전마다 내부 경로가 다를 수 있습니다).")

이 `attn_backend.get_name()`과 `type(attn_layer.impl).__name__`이 우리가 지난번에 코드로 따라갔던 경로와 정확히 일치해야 합니다:

```
LlamaAttention.forward()
  → self.attn (= Attention 레이어, 바로 위에서 찾은 것)
    → self.attn_backend            (예: TritonAttentionBackend)
      → self.impl                  (예: TritonAttentionImpl)
        → unified_attention(...)   (@triton.jit 실제 커널)
```

## 3. 실제 생성해보기

In [ ]:
prompts = [
    "The capital of France is",
    "def fibonacci(n):",
]
sampling_params = SamplingParams(temperature=0.0, max_tokens=32)

outputs = llm.generate(prompts, sampling_params)
for out in outputs:
    print("PROMPT:", out.prompt)
    print("OUTPUT:", out.outputs[0].text)
    print("---")

## 4. 스케줄러/스텝 단위로 들여다보기 (지난번 추적한 파이프라인 확인)

우리가 코드로 따라갔던 `EngineCore.step()`은 매 스텝마다:
1. `scheduler.schedule()` — 이번 배치에 넣을 요청/토큰 결정
2. `model_executor.execute_model()` — 실제 GPU forward
3. `sample_tokens()` — 다음 토큰 샘플링

이걸 직접 들여다보려면 vLLM의 native profiler(`start_profile`/`stop_profile`)를 쓰면 Chrome trace를 만들 수 있습니다. (파일이 꽤 커서 Colab에서 다운로드해 `chrome://tracing`에서 열어봐야 합니다.)

In [ ]:
# torch_profiler_dir은 3번(LLM 생성) 시점에 이미 지정해뒀으므로 여기선 켜고 끄기만 하면 됩니다.
llm.start_profile()
_ = llm.generate(
    ["Explain what a KV cache is in one sentence."],
    SamplingParams(temperature=0.0, max_tokens=32),
)
llm.stop_profile()

print(os.listdir("/content/vllm_trace"))

`/content/vllm_trace`에 `.json` 트레이스 파일이 생깁니다. 다운로드해서 크롬에서 `chrome://tracing`을 열고 드래그해보면, 타임라인 위에 스케줄러/model forward/샘플러가 각각 얼마나 시간을 먹는지 시각적으로 보입니다 — 우리가 코드로만 보던 파이프라인이 실제 시간으로 어디에 쓰이는지 확인하는 거예요.

In [ ]:
from google.colab import files
import glob

trace_files = glob.glob("/content/vllm_trace/*.json")
if trace_files:
    files.download(trace_files[0])
else:
    print("trace 파일이 아직 없습니다 — 위 셀이 에러 없이 끝났는지 확인하세요.")